# EDA — Listado de Retirados (ISTS)

**Fuente:** `data/raw/retiros/retiro.xlsx` (hoja `lista_retirados`)

Es un reporte de exportación con cabecera institucional, título y celdas combinadas antes de la tabla real — típico de un "listado para imprimir" más que de una base de datos tabular. Este notebook documenta cómo limpiarlo y qué tan lejos se puede llegar analíticamente con lo que contiene.

> Parte del Observatorio Institucional de Datos Académicos — ver `PLAN-RETO-OBSERVATORIO.md`, Fase 6.

## 1. Estructura cruda del archivo

Cargamos sin asumir encabezado (`header=None`) para ver la forma real: filas de membrete institucional, un título, y recién en la fila 12 (índice base 0) aparece la fila de encabezado de la tabla.

In [1]:
import pandas as pd

FILE = "../data/raw/retiros/retiro.xlsx"
raw = pd.read_excel(FILE, sheet_name="lista_retirados", header=None)
print("Dimensiones crudas:", raw.shape)
raw.iloc[8:16]

Dimensiones crudas: (18, 19)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
8,NaN,LISTADO DE RETIRADOS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,NaN,GRADO - NOVIEMBRE 2024 - MARZO 2025 MACHALA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,NaN,NaN,RETIRADO,NaN,NaN,CEDULA,NaN,NIVEL,NaN,SEDE,SECCION,NaN,CARRERA,FECHA,MOTIVO,NaN,NaN,NaN,NaN
13,NaN,NaN,FLORES RIVERA NICOLE MIGUEL,NaN,NaN,0748662468,NaN,4TO NIVEL,NaN,SEDE MACHALA,DIURNO_SM,NaN,GASTRONOMIA,2024-12-06 00:00:00,RETIRO POR MOTIVO DE EMBARAZO DE ALTO RIESGO S...,NaN,NaN,NaN,NaN
14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,NaN,NaN,TOTAL DE RETIRADOS: 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Carga limpia

`pandas`/`openpyxl` no expanden celdas combinadas: el valor de una celda fusionada solo aparece en su esquina superior izquierda. Como el título ("RETIRADO", "CEDULA", …) y el único dato real están ambos fusionados de forma consistente, basta con tomar la fila 12 como encabezado y descartar las columnas que quedaron completamente vacías.

In [2]:
df = pd.read_excel(FILE, sheet_name="lista_retirados", header=12, dtype={"CEDULA": str})

# Filas de resumen ("TOTAL DE RETIRADOS: 1") o celdas sueltas solo tienen 1
# valor no nulo disperso entre columnas vacías; una fila de dato real llena
# varias columnas a la vez. Se filtran filas primero y columnas después, para
# que una columna que solo tenía dato en una fila basura también se descarte.
df = df[df.notna().sum(axis=1) >= 3].reset_index(drop=True)
df = df.dropna(axis=1, how="all")
df = df.rename(columns={"RETIRADO": "NOMBRE_RETIRADO"})  # esta columna en realidad trae el nombre, no un Sí/No

print("Dimensiones limpias:", df.shape)
df

Dimensiones limpias: (1, 8)


,NOMBRE_RETIRADO,CEDULA,NIVEL,SEDE,SECCION,CARRERA,FECHA,MOTIVO
0,FLORES RIVERA NICOLE MIGUEL,0748662468,4TO NIVEL,SEDE MACHALA,DIURNO_SM,GASTRONOMIA,2024-12-06,RETIRO POR MOTIVO DE EMBARAZO DE ALTO RIESGO S...


## 3. Limitaciones de esta fuente

Tal como se entregó, este archivo cubre el periodo `GRADO - NOVIEMBRE 2024 - MARZO 2025 MACHALA` y contiene **un único retiro registrado**. Con N = 1 no es posible:

- Calcular una tasa de retiro (no hay denominador de matriculados de ese mismo periodo en este archivo).
- Comparar motivos de retiro entre sí.
- Identificar patrones por carrera, sede, sección o fecha.

El único registro disponible:

In [3]:
cols = ["NIVEL", "SEDE", "SECCION", "CARRERA", "FECHA", "MOTIVO"]
df[cols].T.rename(columns={0: "valor"})

,valor
NIVEL,4TO NIVEL
SEDE,SEDE MACHALA
SECCION,DIURNO_SM
CARRERA,GASTRONOMIA
FECHA,2024-12-06 00:00:00
MOTIVO,RETIRO POR MOTIVO DE EMBARAZO DE ALTO RIESGO S...


## 4. Dónde sí hay volumen: `matriculas.xlsx`

Para analizar retiro con una muestra razonable (N = 40, aunque de un único periodo distinto a este), `matriculas.xlsx` trae sus propias columnas `RETIRADO` / `TIPO RETIRO` por cada inscripción — ver `eda_matriculas.ipynb`, sección 5. En ese periodo la tasa de retiro registrada fue 0%, lo cual **no contradice** el único caso de este archivo: son periodos académicos distintos.

## 5. Hallazgos y recomendación

- Esta fuente, como está hoy, sirve para auditoría puntual de un caso, no para análisis estadístico.
- **Recomendación para el Observatorio:** pedir a la institución un acumulado histórico de `retiro.xlsx` (varios periodos en un solo archivo, o uno por periodo pero consolidable) antes de construir cualquier indicador de "tasa de retiro por motivo/carrera/sede". Con un solo caso, cualquier tablero que muestre un % aquí sería engañoso.
- El campo `MOTIVO` es texto libre extenso (más una nota administrativa que una categoría) — si se acumulan más periodos, conviene definir una categorización cerrada de motivos (ver Fase 5 del plan: diseño de indicadores) para poder agregarlo.

## 6. Exportar dataset limpio

Mismo criterio que en los otros notebooks: se guarda el único registro ya limpio como `.xlsx`, listo para consolidarse con futuros periodos sin repetir el trabajo de desarmar el formato original.

In [4]:
import os

OUT_DIR = "../data/clean"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_FILE = os.path.join(OUT_DIR, "retiro_limpio.xlsx")

df.to_excel(OUT_FILE, sheet_name="lista_retirados", index=False)
print(f"Guardado: {OUT_FILE}  ({df.shape[0]} filas x {df.shape[1]} columnas)")

Guardado: ../data/clean/retiro_limpio.xlsx  (1 filas x 8 columnas)
